# Exploración de Datos (EDA)

***


El objetivo no es hacer modelos complicados, sino aprender a **conocer los datos antes de usarlos**. Esto se llama EDA (*Exploratory Data Analysis*).

### ¿Qué vamos a aprender?

- Cómo identificar qué tipo de variables tenemos
- Cómo detectar valores faltantes y columnas duplicadas
- Cómo construir un *data profiling* de variables numéricas, categóricas y temporales
- Cómo hacer preguntas útiles sobre los datos

### Diccionario de variables



---
## 0. Importar librerías

Antes de trabajar con los datos, necesitamos cargar las herramientas (librerías) que vamos a usar.

- **`pandas`**: la librería principal para manejar tablas de datos en Python. La abreviamos como `pd`.
- **`numpy`**: herramienta para cálculos numéricos. La abreviamos como `np`.

In [ ]:
# estas son paqueterías para análisis en python
import pandas as pd
import numpy as np

print('Librerías cargadas correctamente ✓')

---
## 1. Cargar los datos

Cargamos el archivo CSV con `pd.read_csv()`. El resultado se guarda en un objeto que llamamos `df` (abreviación de *DataFrame*, que es como se llama una tabla en pandas).

In [ ]:
# Ajusta la ruta al lugar donde tengas guardado el archivo
df = pd.read_csv('https://raw.githubusercontent.com/ElenaVillano/eda_geda_conagua/refs/heads/master/data_consumo_agua_updated.csv')

print('Base de datos cargada ✓')
print(f'Tamaño: {df.shape[0]:,} filas  x  {df.shape[1]} columnas')

In [ ]:
df

---
## 2. Primera vista: ¿cómo se ven los datos?

`.head()` muestra las primeras filas. Es lo primero que se hace al abrir cualquier base de datos.

In [ ]:
df.head(3)

In [ ]:
df.tail()

In [ ]:
df.sample(10)

In [ ]:
df.columns

In [ ]:
df.nunique()

---
## 3. Dimensiones de la base

Es la primera pregunta que nos hacemos: ¿qué tan grande es la base?

`df.shape` regresa una tupla con (número de filas, número de columnas).

In [ ]:
filas, columnas = df.shape

print(f'Número de observaciones (filas):  {filas:,}')
print(f'Número de variables (columnas):   {columnas}')

In [ ]:
df.shape

---
## 4. Tipos de variables

Determinar el tipo de variables de las columnas determinará en gran medida el análisis de datos.

Cuatro tipos de variables:

| Tipo | Descripción | Ejemplos en esta base |
|---|---|---|
| **Numéricas** | Valores numéricos (enteros o decimales) | `consumo_total`, `latitud` |
| **Categóricas** | Categorías o grupos | `alcaldia`, `indice_des`, `bimestre` |
| **Temporales** | Fechas o periodos de tiempo | `anio`, `bimestre` (combinados) |
| **Texto libre** | Texto sin estructura fija | (no hay en esta base) |

Pandas detecta automáticamente algunos tipos con `.dtypes`:

In [ ]:
# dtypes muestra el tipo de dato que pandas asignó a cada columna
# object = texto,  int64 = entero,  float64 = decimal
df.dtypes

### 4.1 OJO aquí es tu trabajo determinar qué tipo de variables deben de ser

- ¿Cuáles crees que deben de ser categóricas que Python detectó como numéricas?

- ¿Cuáles crees que son categóricas que no deberían de serlo?

In [ ]:
# Variables numéricas: pandas las detecta como int o float
vars_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Variables categóricas: pandas las detecta como 'object' (texto)
vars_categoricas = df.select_dtypes(include='object').columns.tolist()

print(f'Variables numéricas ({len(vars_numericas)}):  {vars_numericas}')
print()
print(f'Variables categóricas ({len(vars_categoricas)}):  {vars_categoricas}')

In [ ]:
df[vars_categoricas]

In [ ]:
df[vars_numericas]

### 4.2 Cambiar tipos de variables


In [ ]:
df['bimestre'].value_counts()

In [ ]:
# Convert 'bimestre' to string type
df['bimestre_str'] = df['bimestre'].astype(str)

In [ ]:
df

In [ ]:
df['fecha_registro'].value_counts()

In [ ]:
df.dtypes

> **Nota:**
> `gid` es un número entero, pero es un **identificador único** — no tiene sentido calcular su promedio. Hay que tener cuidado con este tipo de columnas.

---
## 5. Valores faltantes (NaN)

Los valores faltantes (`NaN` = *Not a Number*) son celdas vacías. Son muy comunes en datos reales y es importante saber cuántos hay antes de cualquier análisis.

`.isnull().sum()` cuenta los NaN por columna.

In [ ]:
# Conteo de valores faltantes por columna
faltantes = df.isnull().sum()

# También calculamos el porcentaje
pct_faltantes = (faltantes / len(df) * 100).round(2)

# Unimos en una tabla y filtramos solo las que sí tienen faltantes
tabla_faltantes = pd.DataFrame({
    'valores_faltantes': faltantes,
    'porcentaje_%': pct_faltantes
})

tabla_faltantes
#tabla_faltantes[tabla_faltantes['valores_faltantes'] > 0]

---
## 6. Data Profiling: Variables Numéricas

El *data profiling* es un resumen detallado de cada variable. Para las numéricas nos interesa saber: ¿cuál es el rango? ¿hay valores extremos? ¿cómo está distribuida?

Vamos a construirlo paso a paso para entender cada estadístico.

In [ ]:
df.describe()

In [ ]:
variables_numericas = ['consumo_total', 'consumo_prom', 'consumo_total_dom', 'consumo_prom_dom', 'consumo_total_no_dom', 'consumo_prom_no_dom', 'consumo_total_mixto', 'consumo_prom_mixto']

In [ ]:
df[variables_numericas].describe()

**¿Qué buscamos en esta tabla?**

- **Media vs. mediana muy diferentes** → indica valores extremos (outliers). Por ejemplo, si la media de `consumo_total` es mucho mayor que la mediana, hay colonias con consumo excepcionalmente alto que "jalan" el promedio hacia arriba.
- **Desviación estándar muy grande** → los datos están muy dispersos.
- **Mínimo = 0** en columnas de consumo → colonias sin consumo registrado.
- **`anio` con desv_estandar = 0** → todos los registros son del mismo año (2019).

---
## 7. Data Profiling: Variables Categóricas

Para variables categóricas nos interesa saber: ¿cuántas categorías hay? ¿cuál es la más frecuente? ¿están balanceadas o muy desiguales?

In [ ]:
# Variables categóricas de interés (quitamos 'colonia' por tener demasiadas categorías)
vars_cat_analisis = ['alcaldia', 'bimestre', 'indice_des','nomgeo']

profiling_cat = pd.DataFrame(index=vars_cat_analisis)

profiling_cat['n_categorias']  = [df[c].nunique() for c in vars_cat_analisis]
profiling_cat['n_faltantes']   = [df[c].isnull().sum() for c in vars_cat_analisis]
profiling_cat['moda']          = [df[c].mode()[0] for c in vars_cat_analisis]
profiling_cat['top_2']         = [df[c].value_counts().index[1] if df[c].nunique() > 1 else '-' for c in vars_cat_analisis]
profiling_cat['top_3']         = [df[c].value_counts().index[2] if df[c].nunique() > 2 else '-' for c in vars_cat_analisis]

profiling_cat

In [ ]:
df['nomgeo'].value_counts()

In [ ]:
df['bimestre'].value_counts()

## 8. Data Profiling: Variables temporales

Lo primero es ver qué tipo de dato tiene pandas asignado a estas columnas.

Si el resultado dice **`object`** — significa que pandas las está tratando como **texto**, no como fechas.

Eso tiene consecuencias importantes:
- **No puedes restarlas** para calcular duración
- **No puedes ordenarlas** correctamente (el orden de texto es alfabético, no cronológico)
- **No puedes graficarlas** en un eje de tiempo

### ¿En qué formato viene la fecha?

Antes de convertir, necesitamos saber el formato. Los más comunes son:

| Ejemplo en los datos | Formato pandas |
|---|---|
| `2021-03-15` | `'%Y-%m-%d'` |
| `15/03/2021` | `'%d/%m/%Y'` |
| `03/15/2021` | `'%m/%d/%Y'` |
| `March 15, 2021` | `'%B %d, %Y'` |
| `15-Mar-2021` | `'%d-%b-%Y'` |

## 8.1. Convertir a datetime — `pd.to_datetime()`

`pd.to_datetime()` es la función de pandas para convertir texto en fechas reales.  
Una vez convertidas, el tipo de dato será **`datetime64[ns]`** — el formato de fecha de pandas.

#### Conversión automática (cuando el formato es estándar)

In [ ]:
df.dtypes

In [ ]:
df['fecha_registro']

In [ ]:
# Si las fechas vienen en formato ISO (YYYY-MM-DD), pandas las detecta solas
# errors='coerce' convierte los valores que no pueda parsear en NaT (Not a Time)
# en lugar de lanzar un error — útil para detectar fechas mal formateadas
df['fecha_registro_date'] = pd.to_datetime(df['fecha_registro'], errors='coerce')

In [ ]:
df.dtypes

In [ ]:
print("--- Rango de Fechas ---")
min_date = df['fecha_registro_date'].min()
max_date = df['fecha_registro_date'].max()
print(f"Fecha mínima: {min_date}")
print(f"Fecha máxima: {max_date}")


### 8.2 Extraer componentes de la fecha

Una vez que la columna es `datetime`, podemos extraer partes de la fecha con el **accessor `.dt`**.  
Es como abrir la fecha y sacar lo que nos interesa.

| Accessor | Qué devuelve | Rango típico |
|---|---|---|
| `.dt.year` | Año | 2015, 2016... |
| `.dt.month` | Número de mes | 1–12 |
| `.dt.month_name()` | Nombre del mes | 'January'... |
| `.dt.day` | Día del mes | 1–31 |
| `.dt.day_of_week` | Día de la semana | 0=lunes, 6=domingo |
| `.dt.day_name()` | Nombre del día | 'Monday'... |
| `.dt.quarter` | Trimestre | 1–4 |
| `.dt.is_month_start` | ¿Es el primer día del mes? | True/False |

In [ ]:
# Extraemos componentes útiles de la fecha de inicio
df['anio_inicio']     = df['fecha_registro_date'].dt.year
df['mes_inicio']      = df['fecha_registro_date'].dt.month
df['nombre_mes']      = df['fecha_registro_date'].dt.month_name()  # Nombre en inglés
df['trimestre']       = df['fecha_registro_date'].dt.quarter
df['dia_semana']      = df['fecha_registro_date'].dt.day_name()

# Vista de los nuevos componentes
df[['fecha_registro_date', 'anio_inicio', 'mes_inicio', 'nombre_mes', 'trimestre', 'dia_semana']].head(8)

In [ ]:
df['mes_inicio'].value_counts()